In [ ]:
import pandas as pd

In [ ]:
# loading the original csv into a dataframe
water_og = pd.read_csv("/content/Watershed_Surface_Water_Quality_Data_20260317.csv")

In [ ]:
water_og.head()

,ID,Sample Site,Numeric Result,Result Qualifier,Formatted Result,Result Units,Latitude (Degrees),Longitude (Degrees),Sample Date,Parameter,Site Key
0,NaN,Bow River Below Ghost Dam,8.200000,NaN,8.2,pH units,51.22053,-114.70511,2008 May 14 10:35:00 AM,pH,SUR_BR-GD
1,NaN,Bow River Below Ghost Dam,6.500000,NaN,6.5,°C,51.22053,-114.70511,2008 May 14 10:35:00 AM,Water Temperature (Field),SUR_BR-GD
2,NaN,Bow River Below Ghost Dam,302.399994,NaN,302.4,µS/cm,51.22053,-114.70511,2008 May 14 10:35:00 AM,Specific Conductance (Field),SUR_BR-GD
3,NaN,Bow River Below Ghost Dam,12.400000,NaN,12.4,mg/L,51.22053,-114.70511,2008 May 14 10:35:00 AM,Dissolved Oxygen (Field),SUR_BR-GD
4,NaN,Bow River Below Ghost Dam,100.599998,NaN,100.6,%,51.22053,-114.70511,2008 May 14 10:35:00 AM,Dissolved Oxygen (Saturation) (Field),SUR_BR-GD


In [ ]:
water_og.shape

(391401, 11)

In [ ]:
# the ID and result qualifier columns have many missing values, they will be excluded from analysis
water_og.isna().sum()

,0
ID,270639
Sample Site,0
Numeric Result,0
Result Qualifier,299021
Formatted Result,0
Result Units,0
Latitude (Degrees),0
Longitude (Degrees),0
Sample Date,0
Parameter,0


In [ ]:
# checking for all unique qualifiers
water_og['Result Qualifier'].unique()

array([nan, '<', '>'], dtype=object)

In [ ]:
# some values in the formatted result column have a qualifier
# we will remove these qualifiers so the column can be converted to numeric
def remove_qualifier(value):
  chars_to_remove = ['<', '>']
  if value[0] in chars_to_remove:
    return value[1:]
  else:
    return value

water_og['Formatted Result'] = water_og['Formatted Result'].apply(remove_qualifier)

Currently, each chemical tested at each site at each date has its own row in the dataframe. We will pivot the dataframe, so that each row is a site and date with each chemcial having its own column.

In [ ]:
# pivots the table so that each tested chemical is a column
water_og['Formatted Result Numeric'] = pd.to_numeric(water_og['Formatted Result'], errors='coerce')
water_df = water_og.pivot_table(
    index=['Sample Site', 'Latitude (Degrees)', 'Longitude (Degrees)', 'Sample Date', 'Site Key'],
    columns='Parameter',
    values='Formatted Result Numeric',
    aggfunc='mean'
)
water_df.head()

Parameter                                                                                          (Aminomethyl)phosphonic acid  \
Sample Site               Latitude (Degrees) Longitude (Degrees) Sample Date             Site Key                                 
Bearspaw Reservoir Centre 51.117405          -114.292106         1993 Aug 04 12:00:00 AM SUR_BP-C                           NaN   
                                                                 1993 Aug 11 12:00:00 AM SUR_BP-C                           NaN   
                                                                 1993 Aug 18 12:00:00 AM SUR_BP-C                           NaN   
                                                                 1993 Aug 26 12:00:00 AM SUR_BP-C                           NaN   
                                                                 1993 Jul 07 12:00:00 AM SUR_BP-C                           NaN   

Parameter                                                                                          2,4-D  \
Sample Site               Latitude (Degrees) Longitude (Degrees) Sample Date             Site Key          
Bearspaw Reservoir Centre 51.117405          -114.292106         1993 Aug 04 12:00:00 AM SUR_BP-C    NaN   
                                                                 1993 Aug 11 12:00:00 AM SUR_BP-C    NaN   
                                                                 1993 Aug 18 12:00:00 AM SUR_BP-C    NaN   
                                                                 1993 Aug 26 12:00:00 AM SUR_BP-C    NaN   
                                                                 1993 Jul 07 12:00:00 AM SUR_BP-C    NaN   

Parameter                                                                                          2,4-DB  \
Sample Site               Latitude (Degrees) Longitude (Degrees) Sample Date             Site Key           
Bearspaw Reservoir Centre 51.117405          -114.292106         1993 Aug 04 12:00:00 AM SUR_BP-C     NaN   
                                                                 1993 Aug 11 12:00:00 AM SUR_BP-C     NaN   
                                                                 1993 Aug 18 12:00:00 AM SUR_BP-C     NaN   
                                                                 1993 Aug 26 12:00:00 AM SUR_BP-C     NaN   
                                                                 1993 Jul 07 12:00:00 AM SUR_BP-C     NaN   

Parameter                                                                                          2,4-DP  \
Sample Site               Latitude (Degrees) Longitude (Degrees) Sample Date             Site Key           
Bearspaw Reservoir Centre 51.117405          -114.292106         1993 Aug 04 12:00:00 AM SUR_BP-C     NaN   
                                                                 1993 Aug 11 12:00:00 AM SUR_BP-C     NaN   
                                                                 1993 Aug 18 12:00:00 AM SUR_BP-C     NaN   
                                                                 1993 Aug 26 12:00:00 AM SUR_BP-C     NaN   
                                                                 1993 Jul 07 12:00:00 AM SUR_BP-C     NaN   

Parameter                                                                                          2,4-Dichlorophenol  \
Sample Site               Latitude (Degrees) Longitude (Degrees) Sample Date             Site Key                       
Bearspaw Reservoir Centre 51.117405          -114.292106         1993 Aug 04 12:00:00 AM SUR_BP-C                 NaN   
                                                                 1993 Aug 11 12:00:00 AM SUR_BP-C                 NaN   
                                                                 1993 Aug 18 12:00:00 AM SUR_BP-C                 NaN   
                                                                 1993 Aug 26 12:00:00 AM SUR_BP-C                 NaN   
                                                                 1993 J

It appears that, after the pivot, we have lots of missing values. We will create a new dataframe with the proportion of missing values for each column.

In [ ]:
# gets a list of columns from the pivoted dataframe
water_columns = water_df.columns

# gets a list of missing proportions for each column
missing_props = []
for column in water_columns:
  missing_prop = water_df[column].isna().sum() / water_df.shape[0]
  missing_props.append(missing_prop)

# creates a dataframe with the proportion of missing values for each column
water_missing = pd.DataFrame({'column': water_columns, 'missing_prop': missing_props})
water_missing_sorted = water_missing.sort_values(by = "missing_prop")
water_missing_sorted.head(20)

,column,missing_prop
176,Turbidity,0.096954
189,pH,0.106273
148,Specific Conductance (Field),0.151930
168,Total Phosphorus (TP),0.161641
182,Water Temperature (Field),0.163051
162,Total Coliforms,0.210275
15,Ammonia (NH3-N),0.245673
167,Total Organic Carbon (TOC),0.251155
163,Total Dissolved Phosphorus (TDP),0.252408
175,True Color,0.258595


In [ ]:
# exporting the pivoted dataframe to csv
water_df.to_csv("water_pivoted.csv")

In [ ]:
# exporting the csv of missing values to csv
water_missing.to_csv("water_missing_val_by_col.csv")